In [4]:
!pip install -q "vllm>=0.11.0" "transformers>=4.57.0" pyngrok requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 79.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 74.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 110.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 61.6 MB/s eta 0:00:00:

In [ ]:

# ================================================
# Kaggle GPU + vLLM + ngrok Deploy Script
# Qwen2.5-Coder-32B-Instruct (Code LLM)
# ================================================

import os
import subprocess
import time
import sys
from pyngrok import ngrok

# ================== CONFIG ==================
MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"  # Qwen2.5-Coder 32B — Text-only code generation
NGROK_TOKEN = "32bCqPHMLLYElvEw0OT75n4N7vs_4ev6MRxztr1sGfv8PDeJJ"  # Replace with your token!
PORT = 8000

EXTRA_FLAGS = [
    "--gpu-memory-utilization", "0.80",
    "--max-model-len", "32768",
    "--dtype", "auto",
    "--kv-cache-dtype", "fp8_e4m3",
    "--trust-remote-code",
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--served-model-name", "qwen2.5-coder-32b-instruct",
    "--tensor-parallel-size", "1",
    "--disable-log-stats",
]


Starting vLLM for MAI-UI-8B: Tongyi-MAI/MAI-UI-8B


In [ ]:

print(f"Starting vLLM for Qwen2.5-Coder-32B-Instruct: {MODEL_ID}")

import os
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ================== START SERVER ==================
import subprocess, time, requests

cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL_ID] + EXTRA_FLAGS

log_file = open("vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file, text=True)

print("vLLM PID:", proc.pid, "→ tail -f vllm.log to watch")

print("Waiting up to 15 min for /health...")
for i in range(180):
    time.sleep(5)
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
        if r.status_code == 200:
            print("✅ Server is READY!")
            break
    except:
        if i % 6 == 0:
            print(f"({i*5}s) Still waiting... last logs:")
            !tail -n 8 vllm.log
            print("-"*50)

else:
    print("❌ Timeout. Check logs:")
    !tail -n 50 vllm.log
    proc.terminate()
    raise SystemExit("Failed")

# ================== NGROK ==================
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT, "http")
print("\n" + "="*60)
print("🎉 LIVE ENDPOINT:", tunnel.public_url + "/v1")
print("Model name: qwen2.5-coder-32b-instruct")
print("Use in code: OpenAI(base_url=..., api_key='anything')")
print("="*60)

# Keep alive
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.disconnect(tunnel.public_url)
    proc.terminate()


vLLM PID: 214 → tail -f vllm.log to watch
Waiting up to 15 min for /health...
(0s) Still waiting... last logs:
DEBUG 03-01 09:47:40 [platforms/__init__.py:84] Confirmed CUDA platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:112] Checking if ROCm platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:126] ROCm platform is not available because: No module named 'amdsmi'
DEBUG 03-01 09:47:40 [platforms/__init__.py:133] Checking if XPU platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:155] Checking if CPU platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:61] Checking if CUDA platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:84] Confirmed CUDA platform is available.
DEBUG 03-01 09:47:40 [platforms/__init__.py:220] Automatically detected platform cuda.
--------------------------------------------------
(30s) Still waiting... last logs:
(APIServer pid=214) INFO 03-01 09:47:58 [entrypoints/utils.py:287] 
(APIServer

t=2026-03-01T10:35:31+0000 lvl=warn msg="Stopping forwarder" name=http-8000-744e5381-46bf-4746-b781-97ec0844a813 acceptErr="failed to accept connection: Listener closed"


Shutting down...


PyngrokNgrokURLError: ngrok client exception, URLError: [Errno 111] Connection refused

In [ ]:
import json
import time
from openai import OpenAI

print("\n" + "="*80)
print("PURPLE AGENT TEST: Full Output Monitoring")
print("="*80)

# Wait for server to stabilize
print("\n⏳ Waiting 5 seconds for server to stabilize...")
time.sleep(5)

# Initialize client with Ngrok endpoint
client = OpenAI(
    base_url=tunnel.public_url + "/v1",
    api_key="sk-test",
    default_headers={"ngrok-skip-browser-warning": "true"}
)

# Test 1: Simple inference
test_prompt = """You are a spatial reasoning expert for a 3D grid system.
The grid is 9x9 on the x-z plane (x: -400 to 400, z: -400 to 400).
Y-axis is height (50 to 450 in steps of 100).

Task: "Place one purple block on the center (0,0) and one green block to its right."

Generate coordinates in format: Color,x,y,z;Color,x,y,z
Answer:"""

print("\n" + "-"*80)
print("TEST 1: Simple Coordinate Generation")
print("-"*80)
print(f"Prompt:\n{test_prompt}\n")

try:
    print("🔄 Sending request to vLLM server...")
    response = client.completions.create(
        model="qwen2.5-coder-32b-instruct",
        prompt=test_prompt,
        max_tokens=200,
        temperature=0.2,
    )
    
    print("✅ Response received!")
    print(f"\nModel: {response.model}")
    print(f"Stop Reason: {response.choices[0].finish_reason}")
    print(f"\nGenerated Output:\n{response.choices[0].text}")
    print(f"\nTokens Used: {response.usage.completion_tokens} completion, {response.usage.prompt_tokens} prompt")
    
except Exception as e:
    print(f"❌ Error: {str(e)}")
    print("\nServer may still be initializing. Check /health:")
    print(f"Health check: http://127.0.0.1:8000/health")

# Test 2: Chat completions (for Purple Agent)
print("\n" + "-"*80)
print("TEST 2: Chat-based Instruction (Purple Agent Style)")
print("-"*80)

system_prompt = """You are the Purple Agent responsible for converting natural language instructions into 3D grid coordinates.

COORDINATE SYSTEM:
- X-axis: -400 (left) to +400 (right)
- Z-axis: -400 (top) to +400 (bottom/front)
- Y-axis: 50 (ground) to 450 (top) in steps of 100

DIRECTION RULES:
- "left" = X - 100
- "right" = X + 100
- "front" / "in front" = Z + 100
- "behind" / "back" = Z - 100
- "on top" / "stack" = Y + 100

OUTPUT FORMAT: [BUILD];Color,x,y,z;Color,x,y,z
Include ALL blocks (original + new)."""

user_message = """Instruction: "There is a red row at (0,50,0), (-100,50,0), (100,50,0). Extend it by adding two red blocks to the right. Place one block on top of each end."

Current structure: Red,0,50,0;Red,-100,50,0;Red,100,50,0

Please generate the complete coordinate list."""

print(f"System Prompt:\n{system_prompt}\n")
print(f"User Message:\n{user_message}\n")

try:
    print("🔄 Sending chat request to vLLM server...")
    chat_response = client.chat.completions.create(
        model="qwen2.5-coder-32b-instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2,
        max_tokens=500,
    )
    
    print("✅ Chat response received!")
    print(f"\nModel Response:")
    print(chat_response.choices[0].message.content)
    print(f"\nTokens Used: {chat_response.usage.completion_tokens} completion, {chat_response.usage.prompt_tokens} prompt")
    
except Exception as e:
    print(f"❌ Error: {str(e)}")

print("\n" + "="*80)
print("TEST COMPLETE")
print("="*80)
print("\n✅ Server is working! You can now connect your Purple Agent to:")
print(f"   Base URL: {tunnel.public_url}/v1")
print(f"   Model: qwen2.5-coder-32b-instruct")
print("="*80 + "\n")

In [ ]:
print("\n" + "="*80)
print("PURPLE AGENT CONNECTION GUIDE")
print("="*80)

print(f"""
Your vLLM server is now running and accessible via:

📡 ENDPOINT DETAILS:
   Base URL:      {tunnel.public_url}/v1
   Model Name:    qwen2.5-coder-32b-instruct
   API Key:       sk-anything (local inference)

🔧 PYTHON CONNECTION CODE:
```python
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url="{tunnel.public_url}/v1",
    api_key="sk-test",
    default_headers={{"ngrok-skip-browser-warning": "true"}}
)

# Stage 1: Classification
response = await client.chat.completions.create(
    model="qwen2.5-coder-32b-instruct",
    messages=[
        {{"role": "system", "content": "You are a spatial reasoning classifier..."}},
        {{"role": "user", "content": "User instruction here..."}}
    ],
    temperature=0.2,
    max_tokens=1024
)

print("🎯 Classification Result:")
print(response.choices[0].message.content)

# Stage 2: Coordinate Generation
response = await client.chat.completions.create(
    model="qwen2.5-coder-32b-instruct",
    messages=[
        {{"role": "system", "content": "You are a coordinate generator..."}},
        {{"role": "user", "content": "Generate coordinates for..."}}
    ],
    temperature=0.2,
    max_tokens=1024
)

print("📍 Coordinate Output:")
print(response.choices[0].message.content)
```

🌐 ENVIRONMENT VARIABLES (for your Purple Agent):
   Export these in your .env or shell:
   OPENAI_BASE_URL={tunnel.public_url}/v1
   OPENAI_API_KEY=sk-test
   PURPLE_MODEL=qwen2.5-coder-32b-instruct
   OPENAI_TIMEOUT=60

📊 MONITORING:
   1. Check server health:  curl {tunnel.public_url}/health
   2. View logs:            tail -f vllm.log
   3. Monitor GPU:          !nvidia-smi (in cell above)

⚠️  IMPORTANT: The Ngrok free tier requires the ngrok-skip-browser-warning header:
   default_headers={{"ngrok-skip-browser-warning": "true"}}

✅ To test your Purple Agent:
   1. Update your PURPLE_CONTEXT_ID and model settings
   2. Import your purple_openai.agent.PurplePipeline
   3. Call: agent = PurplePipeline.from_env()
   4. Run: response = await agent.handle_message(instruction, context_id)
   5. Outputs will now appear in detail!
""")

print("="*80 + "\n")